# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [15]:
# Write your code below.
%load_ext dotenv
%dotenv


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [16]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [17]:
import os
from glob import glob

# Write your code below.
price_data_dir = os.getenv("PRICE_DATA")

search_pattern = os.path.join(price_data_dir, "**", "*.parquet")
parquet_files = glob(search_pattern, recursive=True)

print(f"Found {len(parquet_files)} parquet files in {parquet_files}")


Found 2045 parquet files in ['../../05_src/data/prices\\ACN\\ACN_2001\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2001\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2003\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2003\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2004\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2004\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2005\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2005\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2006\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2006\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2007\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2007\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2008\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2008\\part.1.parquet', '../../05_src/data/prices\\

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [18]:
# Write your code below.

# Read the parquet files into a Dask DataFrame
dd_df = dd.read_parquet(parquet_files)

def compute_dask_features(df):
    # Sort by Date within each partition to ensure correct chronological order
    df = df.sort_values('Date')
    
    # Calculate lagged values for Close and Adj Close grouped by ticker
    df['Close_lag_1'] = df.groupby('ticker')['Close'].shift(1)
    df['Adj_Close_lag_1'] = df.groupby('ticker')['Adj Close'].shift(1)
    
    # Calculate returns based on Close price
    df['returns'] = (df['Close'] / df['Close_lag_1']) - 1
    
    # Calculate the high-low trading range for the day
    df['hi_lo_range'] = df['High'] - df['Low']
    
    return df

# Apply the feature computation function to each partition of the Dask DataFrame
dd_feat = dd_df.map_partitions(
    compute_dask_features,
    meta=dd_df._meta.assign(
        Close_lag_1='float64',
        Adj_Close_lag_1='float64',
        returns='float64',
        hi_lo_range='float64'
    )
)

# Preview the first few rows of the resulting Dask DataFrame
dd_feat.head()

,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
137315,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,ACN,2001,NaN,NaN,NaN,0.29
137316,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,ACN,2001,15.17,11.404394,-0.010547,0.25
137317,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,ACN,2001,15.01,11.284108,-0.000666,0.46
137318,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,ACN,2001,15.00,11.276587,-0.009333,0.27
137319,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,ACN,2001,14.86,11.171341,0.006057,0.30


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [19]:
# Write your code below.

# Compute the Dask DataFrame to get a Pandas DataFrame
df_feat = dd_feat.compute()

# Sort the DataFrame by ticker and Date to ensure proper order
df_feat = df_feat.sort_values(['ticker', 'Date'])

# Calculate the 10-day moving average of returns for each ticker
df_feat['returns_ma_10'] = df_feat.groupby('ticker')['returns'].transform(lambda x: x.rolling(window=10, min_periods=1).mean())

print(f"Data successfully converted to Pandas DataFrame with computed features. Shape: {df_feat.shape}")
print("Available columns:", df_feat.columns.tolist())
df_feat.head(15)

Data successfully converted to Pandas DataFrame with computed features. Shape: (239659, 15)
Available columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source', 'ticker', 'Year', 'Close_lag_1', 'Adj_Close_lag_1', 'returns', 'hi_lo_range', 'returns_ma_10']


,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,returns_ma_10
137315,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,ACN,2001,NaN,NaN,NaN,0.29,NaN
137316,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,ACN,2001,15.17,11.404394,-0.010547,0.25,-0.010547
137317,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,ACN,2001,15.01,11.284108,-0.000666,0.46,-0.005607
137318,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,ACN,2001,15.00,11.276587,-0.009333,0.27,-0.006849
137319,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,ACN,2001,14.86,11.171341,0.006057,0.30,-0.003623
137320,2001-07-26,14.95,14.99,14.50,14.50,10.900705,6335300.0,ACN.csv,ACN,2001,14.95,11.238999,-0.030100,0.49,-0.008918
137321,2001-07-27,14.51,14.59,14.50,14.51,10.908223,3524000.0,ACN.csv,ACN,2001,14.50,10.900705,0.000690,0.09,-0.007317
137322,2001-07-30,14.50,14.78,14.50,14.70,11.051059,3654300.0,ACN.csv,ACN,2001,14.51,10.908223,0.013094,0.28,-0.004401
137323,2001-07-31,14.71,15.01,14.60,14.96,11.246520,1429000.0,ACN.csv,ACN,2001,14.70,11.051059,0.017687,0.41,-0.001640
137324,2001-08-01,15.00,15.50,14.90,15.50,11.652478,2087900.0,ACN.csv,ACN,2001,14.96,11.246520,0.036096,0.60,0.002553


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

In [20]:
# Was it necessary to convert to pandas to calculate the moving average return?
No, it was not strictly necessary to convert to pandas to calculate the moving average return. Dask provides similar functionality for groupby and rolling operations, which can be applied directly on the Dask DataFrame. However, converting to pandas may have been done for convenience or to leverage pandas' more extensive feature set and ease of use for certain operations.

# Would it have been better to do it in Dask? Why?
Yes, it would have been better to do it in Dask for several reasons:
1. Memory Efficiency & Scalability: Converting to Pandas using .compute() pulls all 239,659 rows directly into the computer's RAM. While this dataset fits in memory easily, real-world financial datasets often contain hundreds of millions of rows. Doing this in Pandas would trigger an out-of-memory (OOM) crash, whereas Dask handles it smoothly by processing data in lazy, parallel chunks.
2. Streamlined Pipeline (Lazy Evaluation): Computing the moving average inside Dask allows the entire sequence of feature calculations (lags, returns, ranges, and moving averages) to be optimized into a single, cohesive execution graph. This graph can be saved efficiently back to distributed storage (like S3 or a cluster) without ever bottlenecks-loading everything into a single machine's memory.
3. Parallel Execution Over-Head Tradeoff: Alternatively, for this specific sample size, Pandas is faster because it bypasses Dask's multi-threading scheduler overhead. However, from a pure big-data architectural standpoint, performing all operations natively within Dask is the superior design choice.


SyntaxError: unterminated string literal (detected at line 2) (200157193.py, line 2)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.